# Visual Classifier — Flexible Fine-Tuning Workbench

This notebook is a **flexible experimentation workbench** for fine-tuning
the `dima806/ai_vs_human_generated_image_detection` ViT model on Apple Silicon.

### How to use

1. **Run sections 0–2** once to set up the environment and load datasets.
2. **Edit the config block** in Section 3, then run the training cell.
3. **Evaluate** in Section 4.
4. **Repeat** — change the config and re-run Section 3 as many times as you want.
   - Swap the training order (julienlucas first, genimage second)
   - Re-run with different hyperparameters
   - Continue from a saved checkpoint or from the base model
   - Add replay from a previous stage

| Dataset | Key | Purpose |
|---------|-----|--------|
| `nebula/GenImage-arrow` | `genimage` | Broaden the model on diverse AI generators |
| `julienlucas/midjourney-dalle-sd-…` | `julienlucas` | Specialise on Midjourney / DALL·E / SD |

The task is **binary classification**: `0 = Real`, `1 = AI-Generated`.

> ⚠️ **Memory:** The M4 has unified memory (16–32 GB). Batch sizes are
> kept small to avoid OOM. Adjust `BATCH_SIZE` if you have more RAM.

> ⚠️ **Large model checkpoints are .gitignored.** Only lightweight deltas or adapters should be committed.

---
## 0 · Shared Configuration
Global constants shared by all cells.  Edit once at the start.

In [1]:
# ── Base model ──
BASE_MODEL = "dima806/ai_vs_human_generated_image_detection"

# ── GenImage streaming subset sizes ──
TRAIN_PER_CLASS = 4800
VAL_PER_CLASS   = 550
TEST_PER_CLASS  = 1000
SEED            = 42

# ── Output paths ──
MODELS_DIR      = "outputs/models"
EVAL_OUTPUT_DIR  = "outputs"

---
## 1 · Environment Setup

In [2]:
import os, sys, torch

# ── GPU / Accelerator check ──
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("✅ MPS (Apple Silicon) available")
    USE_FP16 = False   # fp16 not fully supported on MPS
elif torch.cuda.is_available():
    print(f"✅ CUDA available – {torch.cuda.get_device_name(0)}")
    USE_FP16 = True
else:
    print("⚠️  No GPU – training will be slow on CPU")
    USE_FP16 = False

print(f"Mixed precision (fp16): {USE_FP16}")
print(f"PyTorch version: {torch.__version__}")

✅ MPS (Apple Silicon) available
Mixed precision (fp16): False
PyTorch version: 2.11.0


### Authentication

GenImage-arrow may require a Hugging Face token.

**Do NOT hardcode your token.** Use one of these safe methods:
- Set the `HF_TOKEN` environment variable before launching the notebook.
- Or run `huggingface_hub.login()` / `notebook_login()` interactively.

In [3]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN is None:
    try:
        from huggingface_hub import notebook_login
        notebook_login()          # interactive widget / prompt
        HF_TOKEN = True           # login() stores the token globally
    except Exception:
        print("⚠️  No HF token found. Public datasets will still work,"
              " but gated datasets may fail.")
        HF_TOKEN = None
else:
    print("✅ HF_TOKEN loaded from environment.")

### Imports

In [4]:
from datasets import load_from_disk
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, AutoModelForImageClassification

# Our helper module (lives next to this notebook)
from two_stage_finetuning import (
    get_device,
    load_model_from,
    
    
    run_training_stage,
    evaluate_model,
)
# Weight-delta helpers from the existing visual_classifier module
from visual_classifier import save_weight_delta

DEVICE = get_device()
print(f"Device: {DEVICE}")

Device: mps


---
## 2 · Load Datasets

Run each cell **once** to load the dataset into memory.  
You only need to load the datasets you plan to use — skip the ones you don't need.

### 2a · GenImage (streamed subset)

In [5]:
import os
from datasets import load_from_disk

genimage_path = '../../data/visual/genimage'
try:
    print('Loading GenImage from disk...')
    genimage_ds = load_from_disk(genimage_path)
except FileNotFoundError:
    print('Dataset not found or empty. Fetching...')
    from build_combined_dataset import fetch_genimage, split_dataset_balanced
    genimage_ds = fetch_genimage(12700, seed=SEED)
    genimage_ds = split_dataset_balanced(genimage_ds, seed=SEED)
    os.makedirs(genimage_path, exist_ok=True)
    genimage_ds.save_to_disk(genimage_path)

genimage_ds

Loading GenImage from disk...


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 10160
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 1270
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1270
    })
})

### 2b · julienlucas dataset

In [6]:
import os
from datasets import load_from_disk, concatenate_datasets, DatasetDict

julienlucas_path = '../../data/visual/julienlucas'
try:
    print('Loading julienlucas from disk...')
    julienlucas_ds = load_from_disk(julienlucas_path)
except FileNotFoundError:
    print('Dataset not found or empty. Fetching...')
    from build_combined_dataset import fetch_julienlucas, split_dataset_balanced
    julienlucas_ds = fetch_julienlucas(12700, seed=SEED)
    julienlucas_ds = split_dataset_balanced(julienlucas_ds, seed=SEED)
    os.makedirs(julienlucas_path, exist_ok=True)
    julienlucas_ds.save_to_disk(julienlucas_path)

julienlucas_ds

Loading julienlucas from disk...


Loading dataset from disk:   0%|          | 0/23 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 8556
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 1069
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 1070
    })
})

### 2c · Combine datasets

In [7]:
combined_path = '../../data/visual/combined_dataset'
try:
    print('Checking for combined dataset...')
    combined_ds = load_from_disk(combined_path)
except FileNotFoundError:
    print('Building combined dataset...')
    try:
        g_ds = genimage_ds
    except NameError:
        g_ds = load_from_disk('../../data/visual/genimage')
    
    # Cast julienlucas features to match genimage exactly so they can be concatenated
    julienlucas_ds = julienlucas_ds.cast(g_ds['train'].features)
    
    train_ds = concatenate_datasets([g_ds['train'], julienlucas_ds['train']]).shuffle(seed=SEED)
    val_ds = concatenate_datasets([g_ds['validation'], julienlucas_ds['validation']]).shuffle(seed=SEED)
    test_ds = concatenate_datasets([g_ds['test'], julienlucas_ds['test']]).shuffle(seed=SEED)
    combined_ds = DatasetDict({'train': train_ds, 'validation': val_ds, 'test': test_ds})
    os.makedirs(combined_path, exist_ok=True)
    combined_ds.save_to_disk(combined_path)

combined_ds

Checking for combined dataset...
Building combined dataset...


Casting the dataset:   0%|          | 0/8556 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1069 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1070 [00:00<?, ? examples/s]

Saving the dataset (0/30 shards):   0%|          | 0/18716 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/2339 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/2340 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 18716
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 2339
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2340
    })
})

---
## 3 · 🔧 Training Run

**Edit the config block below**, then run the cell.  
Re-run this cell as many times as you want with different settings.

#### `START_FROM` options
| Value | Meaning |
|-------|---------|
| `"base"` | Fresh start from the original dima806 model |
| `"outputs/models/run_01_genimage"` | Continue from a previously saved checkpoint |
| `"current"` | Continue from whatever model was trained in the last run (still in memory) |

#### `TRAIN_DATASET` / `REPLAY_DATASET` options
| Value | Dataset |
|-------|---------|
| `"genimage"` | GenImage streaming subset (loaded in 2a) |
| `"julienlucas"` | julienlucas Midjourney/DALL·E/SD (loaded in 2b) |
| `"combined"` | Combined GenImage + julienlucas (loaded in 2c) |

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  🔧  TRAINING RUN CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
RUN_NAME         = "run_01_ft_combined"
START_FROM       = "base"
TRAIN_DATASET    = "combined"
REPLAY_DATASET   = None
EPOCHS           = 4
BATCH_SIZE       = 6
LEARNING_RATE    = 3e-5
FREEZE_LAYERS    = 6
AUGMENT          = True
WEIGHT_DECAY     = 0.05
EARLY_STOPPING   = 2
REPLAY_RATIO     = 0.0


In [9]:
import os
# ── Resolve starting model ──
if START_FROM == "current":
    assert 'current_model' in dir() and current_model is not None, \
        "No current_model in memory. Use 'base' or a saved checkpoint path."
    model = current_model
    print(f"📦  Continuing from current in-memory model")
else:
    model, processor = load_model_from(source=START_FROM, device=DEVICE)

# ── Resolve datasets ──
_datasets = {
    "genimage":    genimage_ds    if 'genimage_ds'    in dir() else None,
    "julienlucas": julienlucas_ds if 'julienlucas_ds' in dir() else None,
    "combined": load_from_disk("../../data/visual/combined_dataset") if os.path.exists("../../data/visual/combined_dataset") else None,
}

train_data = _datasets[TRAIN_DATASET]
assert train_data is not None, f"Dataset '{TRAIN_DATASET}' not loaded. Run the loader cell first."

replay_data = None
if REPLAY_DATASET is not None:
    replay_data = _datasets[REPLAY_DATASET]
    assert replay_data is not None, f"Replay dataset '{REPLAY_DATASET}' not loaded. Run its loader cell first."
    replay_data = replay_data["train"]

# ── Output directory ──
run_output_dir = os.path.join(MODELS_DIR, RUN_NAME)

# ── Print summary ──
print(f"\n{'─'*50}")
print(f"  Run:           {RUN_NAME}")
print(f"  Start from:    {START_FROM}")
print(f"  Train on:      {TRAIN_DATASET} ({len(train_data['train'])} train samples)")
print(f"  Replay from:   {REPLAY_DATASET or 'None'}")
print(f"  Epochs:        {EPOCHS}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Freeze layers: {FREEZE_LAYERS or 'None (full fine-tune)'}")
print(f"  Augment:       {AUGMENT}")
print(f"  Output:        {run_output_dir}")
print(f"{'─'*50}\n")

# ── Train ──
current_model, current_trainer = run_training_stage(
    model=model,
    processor=processor,
    train_ds=train_data["train"],
    eval_ds=train_data["validation"],
    output_dir=run_output_dir,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    stage_name=RUN_NAME,
    fp16=USE_FP16,
    replay_ds=replay_data,
    replay_ratio=REPLAY_RATIO,
    freeze_layers=FREEZE_LAYERS,
    augment=AUGMENT,
    weight_decay=WEIGHT_DECAY,
    early_stopping_patience=EARLY_STOPPING,
)

print(f"\n✅  Training complete.  Model stored in `current_model`.")
print(f"    Saved to: {run_output_dir}")
print(f"    To continue training from this model, set START_FROM = 'current' or '{run_output_dir}'")

📦  Loading model from: outputs/models/run_02_ft_genimage


OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'outputs/models/run_02_ft_genimage'. Use `repo_type` argument if needed.

---
## 4 · 📊 Evaluate Current Model

Evaluates `current_model` on whichever test set(s) you choose.  
Edit the config, then run.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  📊  EVALUATION CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
EVAL_ON     = ["genimage", "julienlucas"]   # which test sets to evaluate on
EVAL_PREFIX = RUN_NAME
EVAL_BATCH  = 6                             # batch size for evaluation

In [ ]:
assert 'current_model' in dir() and current_model is not None, \
    "No current_model to evaluate. Run a training cell first."

_test_sets = {
    "genimage":    genimage_ds["test"]    if 'genimage_ds'    in dir() else None,
    "julienlucas": julienlucas_ds["test"] if 'julienlucas_ds' in dir() else None,
}

eval_results = {}
for ds_name in EVAL_ON:
    test_ds = _test_sets.get(ds_name)
    if test_ds is None:
        print(f"⚠️  Skipping {ds_name} — dataset not loaded")
        continue
    print(f"\n{'='*50}")
    print(f"  Evaluating on: {ds_name} ({len(test_ds)} samples)")
    print(f"{'='*50}")

    prefix = f"{EVAL_PREFIX}_on_ds_{ds_name}"
    metrics = evaluate_model(
        model=current_model,
        processor=processor,
        test_ds=test_ds,
        output_prefix=prefix,
        output_dir=EVAL_OUTPUT_DIR,
        batch_size=EVAL_BATCH,
        fp16=USE_FP16,
    )
    eval_results[ds_name] = metrics

# ── Quick comparison table ──
if eval_results:
    print(f"\n{'='*60}")
    print(f"  📋  Summary for: {EVAL_PREFIX}")
    print(f"{'='*60}")
    print(f"  {'Dataset':<15} {'Accuracy':>10} {'F1':>10} {'Precision':>10} {'Recall':>10}")
    print(f"  {'─'*55}")
    for name, m in eval_results.items():
        print(f"  {name:<15} {m['accuracy']:>10.4f} {m['f1']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f}")

---
## 5 · 💾 Save Weight Delta

Save only the **weight differences** from the base model.  
The resulting file is typically ~74 MB (int8 quantised) and stays under GitHub's 100 MB limit.

> This is **full fine-tuning**, not LoRA/PEFT. The delta is not a
> lightweight adapter — it encodes changes across all parameters.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  💾  DELTA CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
DELTA_OUTPUT_PATH = f"./fine_tuned_model_delta/{RUN_NAME}_weight_delta.pt"

In [ ]:
# Reload module to pick up any fixes
import importlib, visual_classifier
importlib.reload(visual_classifier)
from visual_classifier import save_weight_delta

assert 'current_model' in dir() and current_model is not None, \
    "No current_model to save. Run a training cell first."

delta_path, delta_mb = save_weight_delta(
    fine_tuned_model=current_model,
    base_model_name=BASE_MODEL,
    output_path=DELTA_OUTPUT_PATH,
)
print(f"\n✅ Delta saved to '{delta_path}' ({delta_mb:.2f} MB)")

---
## 6 · 🔍 Compare Models on Sample Images

Load any combination of saved models and compare predictions side-by-side.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  🔍  COMPARISON CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
# Each entry: (display_name, model_name_or_path, delta_path_or_None)
MODELS_TO_COMPARE = [
    ("Base",                        "dima806/ai_vs_human_generated_image_detection",    None),
    ("GenImage",                    "outputs/models/run_01_ft_genimage",                None),
    ("GenImage then JulienLucas",   "outputs/models/run_02_ft_genimage_w_julienlucas",  None),
    # ("Delta",      "dima806/ai_vs_human_generated_image_detection", "fine_tuned_model_delta/weight_delta.pt"),
]

SAMPLE_DIR = "../../data/sample_images"

In [ ]:
from visual_classifier import VisualClassifier
from PIL import Image

# ── Load all comparison models ──
loaded_models = []
for display_name, model_path, delta in MODELS_TO_COMPARE:
    print(f"Loading: {display_name} ...")
    vc = VisualClassifier(
        model_name_or_path=model_path,
        delta_path=delta,
    )
    loaded_models.append((display_name, vc))

print(f"\n✅  Loaded {len(loaded_models)} models for comparison\n")

# ── Run predictions on sample images ──
for fname in sorted(os.listdir(SAMPLE_DIR)):
    if fname.startswith("."):
        continue
    img = Image.open(os.path.join(SAMPLE_DIR, fname))
    expected = "Real" if fname.startswith("real") else "AI Generated"

    print(f"\n{fname}")
    print(f"  Expected: {expected}")
    for display_name, vc in loaded_models:
        result = vc.predict(img)
        marker = "✅" if result['prediction'] == expected else "❌"
        print(f"  {marker} {display_name:>15}:  {result['prediction']} ({result['confidence']:.2%})")

---
## 📝 Quick Reference — Example Workflows

### Workflow A: GenImage first, then julienlucas
```
Run 1:  RUN_NAME="run_01_genimage",    START_FROM="base",                        TRAIN_DATASET="genimage"
Run 2:  RUN_NAME="run_02_julienlucas", START_FROM="current",                     TRAIN_DATASET="julienlucas", REPLAY_DATASET="genimage"
```

### Workflow B: julienlucas first, then genimage
```
Run 1:  RUN_NAME="run_01_julienlucas", START_FROM="base",                        TRAIN_DATASET="julienlucas"
Run 2:  RUN_NAME="run_02_genimage",    START_FROM="current",                     TRAIN_DATASET="genimage",    REPLAY_DATASET="julienlucas"
```

### Workflow C: Re-run with different hyperparams
```
Run 1:  RUN_NAME="run_01_genimage_lr2e5",  START_FROM="base",  LEARNING_RATE=2e-5
Run 2:  RUN_NAME="run_01_genimage_lr1e5",  START_FROM="base",  LEARNING_RATE=1e-5
```

### Workflow D: Resume from a saved checkpoint
```
Run 1:  RUN_NAME="run_03_continued",  START_FROM="outputs/models/run_01_genimage",  TRAIN_DATASET="julienlucas"
```